This document is inteded to showcase usage and allow the IBL team to acess example data until we decide about uploading the full dataset. The idea is to iterate and add improvemenets to the structure, suggest visualization and data access patterns that might be desirable for the final documentation.

To use it, you will need to run this notebook in an environment that has both pynwb and remfile installed plus any additional dependencies that we use for analysis such as matplotib, numpy, pandas, etc.

# Locate Examples

In [ ]:
import h5py
import remfile
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient

# Connect to DANDI and get the dandiset
dandiset_id = "000409"
client = DandiAPIClient()
dandiset = client.get_dandiset(dandiset_id, "draft")

# =============================================================================
# Session EIDs for NEW format files (desc-raw / desc-processed)
# =============================================================================

# Complete session with 2 probes - Session 1 (NYU-39, 2021-05-10, angelakilab)
# - Full data: all videos, pose estimation, spike sorting for both probes
TWO_PROBE_SESSION_EID_1 = "6ed57216-498d-48a6-b48b-a243a34710ea"

# Complete session with 2 probes - Session 2 (NYU-39, 2021-05-11, angelakilab)
# - Full data: all videos, pose estimation, spike sorting for both probes
TWO_PROBE_SESSION_EID_2 = "35ed605c-1a1a-47b1-86ff-2b56144f55af"

# Complete session with 1 probe (NYU-46, 2021-06-25, angelakilab)  
# - Full data: all videos, pose estimation, spike sorting for probe01
ONE_PROBE_SESSION_EID = "64e3fb86-928c-4079-865c-b364205b502e"

# Choose which session to use
session_eid = TWO_PROBE_SESSION_EID_1  # Change to TWO_PROBE_SESSION_EID_2 or ONE_PROBE_SESSION_EID

# =============================================================================
# Fetch assets by EID
# =============================================================================

# First, filter assets by EID
session_assets = [asset for asset in dandiset.get_assets() if session_eid in asset.path]

# Then, extract raw and processed files
raw_asset = next((asset for asset in session_assets if "desc-raw" in asset.path), None)
processed_asset = next((asset for asset in session_assets if "desc-processed" in asset.path), None)

print(f"Session EID: {session_eid}")
print(f"\nRaw file:       {raw_asset.path if raw_asset else 'Not found'}")
print(f"Processed file: {processed_asset.path if processed_asset else 'Not found'}")

# Raw NWBFile 
In NWBFile the raw data is stored in a separate file from the processed data. The raw data file contains the acquisition module with the raw ephys data (both AP and LF bands), the link with the Video metadata (which includes properly aligned timestamps), and, when available, the NIDQ board data.

In [ ]:
s3_url = raw_asset.get_content_url(follow_redirects=1, strip_query=False)
file_system = remfile.File(s3_url)
file = h5py.File(file_system, mode="r")

io = NWBHDF5IO(file=file)
nwbfile_raw = io.read()

nwbfile_raw

## Locating and Accessing the AP and LF bands

In PyNWB the raw data is stored in the acquisition module as an ElectricalSeries. We can access the AP bands of each probe as follows and use the HTML representation to display the metada of the ElectricalSeries

In [ ]:
probe_00_ap_series = nwbfile_raw.acquisition["ElectricalSeriesProbe00AP"]
probe_01_ap_series = nwbfile_raw.acquisition["ElectricalSeriesProbe01AP"]

probe_00_ap_series

The LF bands can be accessed in a similar way:

In [ ]:
probe_00_lf_series = nwbfile_raw.acquisition["ElectricalSeriesProbe00LF"]
probe_01_lf_series = nwbfile_raw.acquisition["ElectricalSeriesProbe01LF"]

probe_01_lf_series

Crucially, each of the ElectricalSeries has its own timestamps that are aligned to the common clock used across the entire NWBFile. Moreover, each of the ElectricalSeries is programatically linked to the corresponding probe metadata stored in the electrodes table. For example, we can access the electrodes table for probe 0 as follows:

In [ ]:
probe_01_ap_series.electrodes.to_dataframe()

## Electrode Metadata
The electrodes attribute of an ElectricalSeries is a region of the more general electrodes table stored in the NWBFile. This table contains all the metadata about each of the channels recorded by the probe, including their location in the brain, their position in the probe shank, and other relevant information. A convenient way to access this information is to convert the table to a pandas dataframe as follows:

In [ ]:
electrodes_df = nwbfile_raw.electrodes.to_dataframe()
electrodes_df.columns

Specially useful columns includes 'x', 'y', and 'z' which contain the coordinates of each electrode in the Allen CCF space. Where x is the anterior-posterior axis (AP), y is the dorsal-ventral axis (DV), and z is the medial-lateral axis (ML). In addition, the location column contains the full name of the brain region where each electrode is located. Another useful property are the coordinates `rel_x` and `rel_y` which contain the position of each electrode in the probe shank (relative to the probe) and can be used to visualize the probe geometry.

## Anatomical Location
More detailed anatomical location information is stored in the localization module

In [ ]:
localization = nwbfile_raw.lab_meta_data["localization"]
localization

These module contains two coordinate tables linking the electrodes to their anatomical locations in the Allen CCF space and in the IBL virtual bregma centered space. We can access these tables as follows:

In [ ]:
ccf_table_df = localization.anatomical_coordinates_tables["ElectrodesCCFv3"].to_dataframe()
ibl_bregma_table_df = localization.anatomical_coordinates_tables["ElectrodesIBLBregma"].to_dataframe()

ccf_table_df.columns

The tables provide the localization of each electrode in different coordinate systems and can be used for more detailed anatomical analyses and visualizations. For example, both tables contain the `probe_name` column which can be used to identify the probe associated with each electrode. In addition, the `ElectrodesIBLBregma` contains the localization in the Cosmos and Beryl spaces that should facilitate certain type of analyses and visualizations.

Both the electrode and the anatomical coordinates tables are present both nwbfiles (the one with the raw data and the one with the processed data) and are identical in both files.

Finally, to show how to combine this information we can provide visualization that combines the relative positions of the electrodes plus the atlas information so we can what areas does the probe traverse. We use here the Cosmos parcellation for this visualization as that provides an overall and easy view of the probe localization.

In [ ]:
import matplotlib.pyplot as plt

from ibl_to_nwb.utils import COSMOS_FULL_NAMES, get_cosmos_color

nwbfile = nwbfile_raw

# %% Get anatomical localization table
anatomical_tables = nwbfile.lab_meta_data["localization"].anatomical_coordinates_tables
ibl_bregma_table_df = anatomical_tables["ElectrodesIBLBregma"].to_dataframe()

# %% Get all unique probe names
probe_names = sorted(ibl_bregma_table_df["probe_name"].unique())
n_probes = len(probe_names)

print(f"Found {n_probes} probe(s): {probe_names}")

# %% Create figure - 2 columns per probe (contacts + regions), side by side
fig, axes = plt.subplots(1, n_probes * 2, figsize=(5 * n_probes, 12), sharey=True)

# Handle single probe case (axes won't be a list)
if n_probes == 1:
    axes = [axes[0], axes[1]]

# Track all unique regions across all probes for the legend
all_unique_regions = []

# %% Plot each probe
for probe_idx, probe_name in enumerate(probe_names):
    # Get axes for this probe
    ax_probe = axes[probe_idx * 2]
    ax_regions = axes[probe_idx * 2 + 1]

    # Filter data for this probe
    probe_table = ibl_bregma_table_df[ibl_bregma_table_df["probe_name"] == probe_name].copy()

    # Extract electrode positions from localized_entity
    probe_table["rel_x"] = [electrode["rel_x"].iloc[0] for electrode in probe_table["localized_entity"]]
    probe_table["rel_y"] = [electrode["rel_y"].iloc[0] for electrode in probe_table["localized_entity"]]

    # Sort by depth (rel_y)
    probe_table = probe_table.sort_values("rel_y").reset_index(drop=True)

    # Precompute colors
    cosmos_values = probe_table["cosmos_location"].values
    rel_x_values = probe_table["rel_x"].values
    rel_y_values = probe_table["rel_y"].values
    colors = [get_cosmos_color(c) for c in cosmos_values]

    # Track unique regions for legend
    for region in cosmos_values:
        if region not in all_unique_regions:
            all_unique_regions.append(region)

    # --- Left panel: Probe contacts ---
    contact_width = 12  # um
    contact_height = 10  # um

    for i in range(len(rel_x_values)):
        rect = plt.Rectangle(
            (rel_x_values[i] - contact_width / 2, rel_y_values[i] - contact_height / 2),
            contact_width,
            contact_height,
            facecolor=colors[i],
            edgecolor="black",
            linewidth=0.3,
        )
        ax_probe.add_patch(rect)

    # Set probe axis limits
    x_min, x_max = probe_table["rel_x"].min(), probe_table["rel_x"].max()
    y_min, y_max = probe_table["rel_y"].min(), probe_table["rel_y"].max()
    x_margin, y_margin = 50, 100

    ax_probe.set_xlim(x_min - x_margin, x_max + x_margin)
    ax_probe.set_ylim(y_min - y_margin, y_max + y_margin)
    
    # Simplify x-axis: single tick at max value
    ax_probe.set_xticks([x_max])
    ax_probe.set_xticklabels([f"{x_max:.0f}"])
    ax_probe.set_xlabel("Relative X (um)")
    
    if probe_idx == 0:
        ax_probe.set_ylabel("Distance to Probe Tip (um)")
    ax_probe.set_title(f"{probe_name}")
    ax_probe.set_aspect("equal", adjustable="box")

    # --- Right panel: Region blocks ---
    blocks = []
    current_region = cosmos_values[0]
    block_start = rel_y_values[0]

    for i in range(1, len(cosmos_values)):
        if cosmos_values[i] != current_region:
            block_end = (rel_y_values[i - 1] + rel_y_values[i]) / 2
            blocks.append((current_region, block_start, block_end))
            current_region = cosmos_values[i]
            block_start = block_end

    # Add final block
    blocks.append((current_region, block_start, rel_y_values[-1] + contact_height / 2))

    # Plot region blocks
    block_width = 1.0
    for region, y_start, y_end in blocks:
        color = get_cosmos_color(region)
        height = y_end - y_start

        rect = plt.Rectangle(
            (0, y_start),
            block_width,
            height,
            facecolor=color,
            edgecolor="black",
            linewidth=0.5,
        )
        ax_regions.add_patch(rect)

        # Add label in the middle of the block
        y_center = (y_start + y_end) / 2
        display_name = COSMOS_FULL_NAMES.get(region, region)
        ax_regions.text(block_width + 0.1, y_center, display_name, va="center", ha="left", fontsize=9, fontweight="bold")

    ax_regions.set_xlim(-0.2, 2.0)
    ax_regions.set_ylim(y_min - y_margin, y_max + y_margin)
    ax_regions.set_xticks([])
    ax_regions.set_title("Cosmos Regions")
    ax_regions.axis("off")

# %% Add legend and title
legend_handles = [
    plt.Rectangle((0, 0), 1, 1, facecolor=get_cosmos_color(r), edgecolor="black", linewidth=0.5)
    for r in all_unique_regions
]
legend_labels = [COSMOS_FULL_NAMES.get(r, r) for r in all_unique_regions]
# fig.legend(
#     legend_handles,
#     legend_labels,
#     loc="lower center",
#     ncol=min(5, len(all_unique_regions)),
#     title="Cosmos Regions",
#     bbox_to_anchor=(0.5, 0.02),
# )

plt.suptitle(f"Session: {nwbfile.session_id}", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0.08, 1, 0.96])

plt.show()

## Localization of Probes in the CCF space
To get an overview of where the probes are located in the brain we can use brainrender to visualize the probe shanks together with the brain regions they traverse. Here is an example of how to do this for one of the probes using the Cosmos parcellation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from brainglobe_atlasapi import BrainGlobeAtlas

# %% Get CCFv3 coordinates for all probes
anatomical_tables = nwbfile_raw.lab_meta_data["localization"].anatomical_coordinates_tables
ccf_table_df = anatomical_tables["ElectrodesCCFv3"].to_dataframe()

# Get all unique probe names
probe_names = sorted(ccf_table_df["probe_name"].unique())
n_probes = len(probe_names)

# Define colors for each probe
probe_colors = ["red", "blue", "green", "orange", "purple"][:n_probes]

# %% Load atlas
atlas = BrainGlobeAtlas("allen_mouse_25um")

# %% Collect probe data - compute slice positions per probe (mean coordinate approach)
# See documentation/probe_slice_visualization_guide.md for heuristics
probe_data = {}

for probe_name in probe_names:
    probe_table = ccf_table_df[ccf_table_df["probe_name"] == probe_name].copy()
    bg_x = probe_table["x"].values  # Anterior (0) to Posterior (13200)
    bg_y = probe_table["y"].values  # Dorsal (0) to Ventral (8000)
    bg_z = probe_table["z"].values  # Left (0) to Right (11400)
    
    # Calculate slice positions for this specific probe (mean coordinate)
    coronal_position = int(np.mean(bg_x))
    sagittal_position = int(np.mean(bg_z))
    horizontal_position = int(np.mean(bg_y))
    
    # Convert to slice indices
    coronal_index = int(coronal_position / atlas.resolution[0])
    sagittal_index = int(sagittal_position / atlas.resolution[2])
    horizontal_index = int(horizontal_position / atlas.resolution[1])
    
    probe_data[probe_name] = {
        "x_px": bg_x / atlas.resolution[0],
        "y_px": bg_y / atlas.resolution[1],
        "z_px": bg_z / atlas.resolution[2],
        "coronal_position": coronal_position,
        "sagittal_position": sagittal_position,
        "horizontal_position": horizontal_position,
        "coronal_index": coronal_index,
        "sagittal_index": sagittal_index,
        "horizontal_index": horizontal_index,
    }

# %% Create Figure: Grid with n_probes rows x 3 columns (one row per probe)
fig, axes = plt.subplots(n_probes, 3, figsize=(20, 7 * n_probes))

# Handle single probe case (axes won't be 2D)
if n_probes == 1:
    axes = axes.reshape(1, -1)

for row, probe_name in enumerate(probe_names):
    data = probe_data[probe_name]
    color = probe_colors[row]
    
    # Get slices centered on this probe's location
    reference_coronal = atlas.reference[data["coronal_index"], :, :]
    reference_sagittal = atlas.reference[:, :, data["sagittal_index"]]
    reference_horizontal = atlas.reference[:, data["horizontal_index"], :]
    
    # Coronal view
    ax = axes[row, 0]
    ax.imshow(reference_coronal, cmap="gray", aspect="equal")
    ax.scatter(data["z_px"], data["y_px"], c=color, s=10, alpha=0.8)
    ax.set_xlabel("Left - Right (pixels)")
    ax.set_ylabel("Dorsal - Ventral (pixels)")
    ax.set_title(f"{probe_name} - Coronal (AP = {data['coronal_position']} um)")
    
    # Sagittal view (transposed for correct orientation)
    ax = axes[row, 1]
    ax.imshow(reference_sagittal.T, cmap="gray", aspect="equal")
    ax.scatter(data["x_px"], data["y_px"], c=color, s=10, alpha=0.8)
    ax.set_xlabel("Anterior - Posterior (pixels)")
    ax.set_ylabel("Dorsal - Ventral (pixels)")
    ax.set_title(f"{probe_name} - Sagittal (ML = {data['sagittal_position']} um)")
    
    # Horizontal view
    ax = axes[row, 2]
    ax.imshow(reference_horizontal, cmap="gray", aspect="equal")
    ax.scatter(data["z_px"], data["x_px"], c=color, s=10, alpha=0.8)
    ax.set_xlabel("Left - Right (pixels)")
    ax.set_ylabel("Anterior - Posterior (pixels)")
    ax.set_title(f"{probe_name} - Horizontal (DV = {data['horizontal_position']} um)")

fig.suptitle(f"Session: {nwbfile_raw.session_id} - Probe Trajectories (Reference)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Processed files

In [ ]:
s3_url = processed_asset.get_content_url(follow_redirects=1, strip_query=False)
file_system = remfile.File(s3_url)
file = h5py.File(file_system, mode="r")

io = NWBHDF5IO(file=file)
nwbfile_processed = io.read()

nwbfile_processed

In [ ]:
nwbfile_processed.session_start_time

In [ ]:
nwbfile_processed.subject

In [ ]:
len(nwbfile_processed.trials)

In [ ]:
units_df = nwbfile_processed.units.to_dataframe()
trials_df = nwbfile_processed.trials.to_dataframe()
electrodes_df = nwbfile_processed.electrodes.to_dataframe()

In [ ]:
trials_df.shape

In [ ]:
units_df.shape

In [ ]:
units_probe_00_df = units_df[units_df["probe_name"] == "Probe00"]
units_probe_00_df.shape

In [ ]:
units_df["probe_name"].unique()

In [ ]:
units_df.columns

In [ ]:
units_df.columns

In [ ]:
nwbfile_processed.trials.to_dataframe()

In [ ]:
nwbfile_processed.units.to_dataframe().columns

In [ ]:
nwbfile_processed.units.to_dataframe()